# Semantic Embedding Extraction — PE File Dataset

This notebook generates **384-dimensional sentence embeddings** from static PE (Portable Executable)
file features using `paraphrase-MiniLM-L6-v2`.

> **Note:** Model can be swapped to `all-MiniLM-L6-v2` after further testing — both produce 384-dim vectors.

**Approach:** Selected PE metadata columns (imports, exports, section entropy, resources, etc.) are
concatenated into a single string per sample. The string is then encoded by the transformer,
producing a dense vector used by the downstream Semantic Agent for ransomware classification.

**Input:** `/mnt/4TBSTORE/Dr_Kabuye4TB/RDset/Ransomware.csv` (pipe-delimited, PE static features)  
**Output:** `semantic_dataset_with_embeddings.parquet` — same rows + `semantic_text` + `embedding` (384-dim)

### Labelling
The raw `legitimate` column uses inverted encoding:
- `legitimate = 1` → benign  → `label = 0`
- `legitimate = 0` → ransomware → `label = 1`

### Semantic feature columns used
These PE metadata fields are joined as text for embedding:
`Name`, `Machine`, `Characteristics`, `ImportsNbDLL`, `ImportsNb`, `ExportsNb`,
`SectionsMeanEntropy`, `SectionsMaxEntropy`, `SectionsMinEntropy`,
`ResourcesNb`, `ResourcesMeanEntropy`, `ResourcesMinEntropy`, `ResourcesMaxEntropy`,
`ResourcesMeanSize`, `ResourcesMinSize`, `ResourcesMaxSize`,
`LoadConfigurationSize`, `VersionInformationSize`, `Subsystem`, `DllCharacteristics`

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_CSV      = "/mnt/4TBSTORE/Dr_Kabuye4TB/RDset/Ransomware.csv"
OUTPUT_PARQUET = "/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet"

# ── Model ────────────────────────────────────────────────────────────────────
# NOTE: swap to 'sentence-transformers/all-MiniLM-L6-v2' after further testing
MODEL_NAME = "paraphrase-MiniLM-L6-v2"
BATCH_SIZE = 256

# ── Columns to drop before modelling ─────────────────────────────────────────
# These are identifiers, raw header fields, or memory layout constants
# that carry no semantic signal for ransomware detection
DROP_COLS = [
    "md5", "legitimate",
    "SizeOfOptionalHeader", "MajorLinkerVersion", "MinorLinkerVersion",
    "BaseOfCode", "BaseOfData", "ImageBase",
    "SectionAlignment", "FileAlignment", "CheckSum",
    "SizeOfHeapReserve", "SizeOfHeapCommit",
    "SizeOfStackReserve", "SizeOfStackCommit",
    "LoaderFlags", "NumberOfRvaAndSizes",
]

# ── Semantic columns to embed ─────────────────────────────────────────────────
# PE metadata fields that describe the file's structure and behaviour profile
SEMANTIC_COLUMNS = [
    "Name", "Machine", "Characteristics",
    "ImportsNbDLL", "ImportsNb", "ExportsNb",
    "SectionsMeanEntropy", "SectionsMaxEntropy", "SectionsMinEntropy",
    "ResourcesNb", "ResourcesMeanEntropy", "ResourcesMinEntropy",
    "ResourcesMaxEntropy", "ResourcesMeanSize", "ResourcesMinSize",
    "ResourcesMaxSize", "LoadConfigurationSize", "VersionInformationSize",
    "Subsystem", "DllCharacteristics",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {device}")
print(f"Input          : {INPUT_CSV}")
print(f"Output         : {OUTPUT_PARQUET}")
print(f"Model          : {MODEL_NAME}")

## 2. Load and Prepare the Dataset

In [ ]:
df = pd.read_csv(INPUT_CSV, delimiter="|", engine="python")
print(f"Loaded         : {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns        : {df.columns.tolist()}")
print(f"\nRaw label distribution (legitimate column):")
print(df["legitimate"].value_counts())

In [ ]:
# Invert the label: legitimate=1 means benign (0), legitimate=0 means ransomware (1)
df["label"] = df["legitimate"].apply(lambda x: 0 if x == 1 else 1)

print(f"Label distribution after inversion:")
print(df["label"].value_counts())
print(f"  0 = benign, 1 = ransomware")

# Drop low-signal and identifier columns
df = df.drop(columns=DROP_COLS, errors="ignore")
print(f"\nShape after dropping {len(DROP_COLS)} columns: {df.shape}")

df.head(3)

## 3. Build the Semantic Text Field

The 20 PE metadata columns are cast to strings and joined with a space.
This gives the transformer a compact, structured sentence per file
(e.g. `"calc.exe 332 8448 12 57 0 6.523 7.112 5.901 …"`).

Only columns that are actually present in the loaded CSV are used
(handles datasets where some fields may be missing).

In [ ]:
# Keep only columns that exist in this CSV
present_semantic_cols = [c for c in SEMANTIC_COLUMNS if c in df.columns]
missing = set(SEMANTIC_COLUMNS) - set(present_semantic_cols)
if missing:
    print(f"[INFO] {len(missing)} semantic columns not found in CSV (will be skipped): {missing}")

df["semantic_text"] = df[present_semantic_cols].astype(str).agg(" ".join, axis=1)

print(f"Using {len(present_semantic_cols)} semantic columns.")
print(f"\nSample semantic_text entries:")
for i, txt in enumerate(df["semantic_text"].iloc[:3]):
    print(f"  [{i}] {txt[:120]}")

## 4. Load the Sentence Transformer Model

In [ ]:
print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device=str(device))

print(f"Running on   : {device}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 5. Generate Embeddings

In [ ]:
texts = df["semantic_text"].tolist()
print(f"Encoding {len(texts):,} samples (batch_size={BATCH_SIZE})...")

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)  # [N, 384]

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"dtype                  : {embeddings.dtype}")
print(f"Mean L2 norm           : {np.linalg.norm(embeddings, axis=1).mean():.4f}")

## 6. Attach Embeddings to the DataFrame

In [ ]:
df["embedding"] = embeddings.tolist()

print(f"Final DataFrame shape  : {df.shape}")
print(f"Embedding vector length: {len(df['embedding'].iloc[0])}")
df[["label", "semantic_text", "embedding"]].head(2)

## 7. Sanity Check — Embedding Separation

A quick cosine similarity check to confirm benign and ransomware embeddings
are separable in the embedding space.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

benign_emb = embeddings[df["label"].values == 0][:100]
ransom_emb = embeddings[df["label"].values == 1][:100]

if len(benign_emb) >= 2 and len(ransom_emb) >= 2:
    sim_within_benign = cosine_similarity(benign_emb).mean()
    sim_within_ransom = cosine_similarity(ransom_emb).mean()
    sim_cross         = cosine_similarity(benign_emb, ransom_emb).mean()

    print(f"Within-benign similarity    : {sim_within_benign:.4f}")
    print(f"Within-ransomware similarity: {sim_within_ransom:.4f}")
    print(f"Cross-class similarity      : {sim_cross:.4f}")
    print("\n(Lower cross-class = better separation)")
else:
    print("[INFO] Not enough samples for similarity check.")

## 8. Save to Parquet

In [ ]:
Path(OUTPUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved {len(df):,} rows → {OUTPUT_PARQUET}")

# Reload and verify
verify = pd.read_parquet(OUTPUT_PARQUET)
assert verify.shape == df.shape, f"[WARN] Shape mismatch: {verify.shape} != {df.shape}"
assert len(verify["embedding"].iloc[0]) == 384, "[WARN] Unexpected embedding dimension!"
print(f"Reload check   : {verify.shape} — OK")
print(f"Columns saved  : {verify.columns.tolist()}")